# Minimal Friend DMLPA notebook for SCRESIA

This notebook keeps the original DMLPA model architecture from the prior notebooks as intact as possible. It changes only the execution scaffolding needed to run cleanly in Google Colab or Kaggle:

- clone/install the `scresia` repository before importing it;
- install missing runtime packages, including `einops`;
- reset the vectorized environment before stepping/training;
- save model, normalization statistics, and evaluation metrics as reproducible artifacts.

The DMLPA class below is intentionally kept in the same style and structure as the original notebook.

In [ ]:
# =====================
# 0) Minimal run config
# =====================

RUN_PROFILE = "debug"  # "debug" for a quick smoke test; "serious" for a full run.

GIT_URL = "https://github.com/Thom-320/scres-ia.git"
GIT_BRANCH = "main"
REPO_DIR_COLAB = "/content/scresia"
REPO_DIR_KAGGLE = "/kaggle/working/scresia"

LOCAL_OUTPUT_DIR = "/content/friend_dmlpa_minimal_outputs"
KAGGLE_OUTPUT_DIR = "/kaggle/working/friend_dmlpa_minimal_outputs"

RUNTIME_PIP_PACKAGES = [
    "simpy>=4.1",
    "numpy>=1.26",
    "gymnasium>=0.29",
    "stable-baselines3>=2.3",
    "torch>=2.1",
    "einops>=0.7",
    "pandas>=2.2",
]

SEED = 42
FRAME_STACK = 30
FEATURES_DIM = 120

REWARD_MODE = "ReT_cd_v1"
RISK_LEVEL = "increased"
STOCHASTIC_PT = True
STEP_SIZE_HOURS = 168

TOTAL_TIMESTEPS_DEBUG = 512
MAX_STEPS_DEBUG = 40
EVAL_EPISODES_DEBUG = 1

TOTAL_TIMESTEPS_SERIOUS = 100_000
MAX_STEPS_SERIOUS = 10_000
EVAL_EPISODES_SERIOUS = 5

N_STEPS_DEBUG = 128
BATCH_SIZE_DEBUG = 64
N_EPOCHS_DEBUG = 2
N_STEPS_SERIOUS = 1024
BATCH_SIZE_SERIOUS = 256
N_EPOCHS_SERIOUS = 10
DEVICE = "auto"

if RUN_PROFILE == "debug":
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_DEBUG
    MAX_STEPS = MAX_STEPS_DEBUG
    EVAL_EPISODES = EVAL_EPISODES_DEBUG
    N_STEPS = N_STEPS_DEBUG
    BATCH_SIZE = BATCH_SIZE_DEBUG
    N_EPOCHS = N_EPOCHS_DEBUG
else:
    TOTAL_TIMESTEPS = TOTAL_TIMESTEPS_SERIOUS
    MAX_STEPS = MAX_STEPS_SERIOUS
    EVAL_EPISODES = EVAL_EPISODES_SERIOUS
    N_STEPS = N_STEPS_SERIOUS
    BATCH_SIZE = BATCH_SIZE_SERIOUS
    N_EPOCHS = N_EPOCHS_SERIOUS

print({
    "RUN_PROFILE": RUN_PROFILE,
    "TOTAL_TIMESTEPS": TOTAL_TIMESTEPS,
    "MAX_STEPS": MAX_STEPS,
    "EVAL_EPISODES": EVAL_EPISODES,
    "N_STEPS": N_STEPS,
    "BATCH_SIZE": BATCH_SIZE,
    "N_EPOCHS": N_EPOCHS,
})

In [ ]:
# =======================================
# 1) Portable Colab/Kaggle repository setup
# =======================================

import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IN_KAGGLE = Path("/kaggle/working").exists()
if not (IN_COLAB or IN_KAGGLE):
    # Local macOS-only validation workaround for duplicate OpenMP runtimes.
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-2000:])
    if result.stderr:
        print(result.stderr[-2000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {cmd}")
    return result

# Install missing packages first. Kaggle often lacks stable-baselines3; Colab often lacks simpy/einops.
# Local Homebrew Python may be externally managed, so local runs assume dependencies are already installed.
if IN_COLAB or IN_KAGGLE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", *RUNTIME_PIP_PACKAGES])
else:
    print("local run: skipping pip install")

if IN_KAGGLE:
    REPO_DIR = Path(REPO_DIR_KAGGLE)
    output_root = Path(KAGGLE_OUTPUT_DIR)
elif IN_COLAB:
    REPO_DIR = Path(REPO_DIR_COLAB)
    output_root = Path(LOCAL_OUTPUT_DIR)
else:
    # Local fallback for development on this Mac. Colab/Kaggle will use /content or /kaggle/working.
    REPO_DIR = Path.cwd()
    output_root = Path.cwd() / "outputs" / "friend_dmlpa_minimal"

if IN_COLAB or IN_KAGGLE:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    run_cmd(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO_DIR)])
    # `scresia.supply_chain...` needs the parent; the repo path also protects older internal absolute imports.
    sys.path.insert(0, str(REPO_DIR.parent))
    sys.path.insert(0, str(REPO_DIR))
else:
    # Local fallback: this repo directory is not named scresia on Thommy's Mac.
    sys.path.insert(0, str(REPO_DIR.parent))
    sys.path.insert(0, str(REPO_DIR))

output_root.mkdir(parents=True, exist_ok=True)
print("repo:", REPO_DIR)
print("outputs:", output_root)
print("python path head:", sys.path[:3])

In [ ]:
# ==========================
# 2) Imports
# ==========================

import json
import time

import numpy as np
import pandas as pd
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecNormalize
from einops import rearrange, repeat

try:
    from scresia.supply_chain.external_env_interface import (
        get_episode_terminal_metrics,
        make_dkana_thesis_faithful_env,
    )
except ModuleNotFoundError:
    # Local-only fallback when developing from a checkout whose folder is not named `scresia`.
    from supply_chain.external_env_interface import (
        get_episode_terminal_metrics,
        make_dkana_thesis_faithful_env,
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
print("sb3 device setting:", DEVICE)

In [ ]:
# ==============================================
# 3) Environment: same default DKANA thesis env
# ==============================================

# This intentionally keeps the default DKANA thesis decision contract used by the earlier notebooks:
# - action space: Box(18), the raw thesis decision vector
# - observation before frame stack: 19 dimensions
# - VecFrameStack(30), so DMLPA sees 30 temporal frames

def make_env(seed=SEED):
    def _init():
        env = make_dkana_thesis_faithful_env(
            reward_mode=REWARD_MODE,
            risk_level=RISK_LEVEL,
            stochastic_pt=STOCHASTIC_PT,
            max_steps=MAX_STEPS,
            step_size_hours=STEP_SIZE_HOURS,
        )
        env.reset(seed=seed)
        return Monitor(env)
    return _init

venv = DummyVecEnv([make_env(SEED)])
env = VecFrameStack(venv, n_stack=FRAME_STACK)
env = VecNormalize(env, norm_reward=True, norm_obs=True)

obs = env.reset()
print("stacked observation shape:", obs.shape)
print("observation space:", env.observation_space)
print("action space:", env.action_space)

assert env.action_space.shape == (18,), env.action_space
assert env.observation_space.shape[0] % FRAME_STACK == 0, env.observation_space

## Friend DMLPA architecture

The class below preserves the original model shape and operations from `Untitled90.ipynb` and `notebookffc7b2c5ff.ipynb`: `Linear -> GELU -> Linear -> TransformerEncoder -> last token`. The only surrounding changes are outside the class: portable setup, clean reset/training, and artifact export.

In [ ]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from einops import rearrange,repeat



class DMLPA(BaseFeaturesExtractor):
  def __init__(self,observation_space,factor,features_dim=120):
    super().__init__(observation_space,features_dim)
    self.obs_dimension=observation_space.shape[0]//factor
    #print(self.obs_dimension)

    self.latent_rw=torch.nn.Sequential(
        torch.nn.Linear(self.obs_dimension,100),
        torch.nn.GELU(),
        torch.nn.Linear(100,features_dim))

    self.attlayer=torch.nn.TransformerEncoderLayer(d_model=features_dim,nhead=12,batch_first=True)
    self.accumulated=torch.nn.TransformerEncoder(self.attlayer,num_layers=4)
  def forward(self,observations):
    batch_size=observations.shape[0]
    observations=rearrange(observations,"b (d k) -> b d k",k=self.obs_dimension)
    #print(observations.shape)

    observations=self.latent_rw(observations)
    observations=self.accumulated(observations)
    observations=observations[:,-1,:]
    #print(observations.shape)
    return observations

In [ ]:
# ==============================================
# 4) Quick architecture shape check
# ==============================================

extractor = DMLPA(env.observation_space, factor=FRAME_STACK, features_dim=FEATURES_DIM)
with torch.no_grad():
    sample = torch.as_tensor(obs, dtype=torch.float32)
    features = extractor(sample)
print("DMLPA obs_dimension:", extractor.obs_dimension)
print("DMLPA feature shape:", tuple(features.shape))
assert features.shape[-1] == FEATURES_DIM

In [ ]:
# ==============================================
# 5) PPO with friend DMLPA feature extractor
# ==============================================

policy_kwargs = dict(
    features_extractor_class=DMLPA,
    features_extractor_kwargs=dict(features_dim=FEATURES_DIM,factor=FRAME_STACK),
)

model = PPO(
    "MlpPolicy",
    env,
    policy_kwargs=policy_kwargs,
    device=DEVICE,
    verbose=1,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    seed=SEED,
)

start = time.time()
model.learn(total_timesteps=TOTAL_TIMESTEPS)
elapsed_seconds = time.time() - start
print("training seconds:", elapsed_seconds)

In [ ]:
# ==============================================
# 6) Deterministic evaluation
# ==============================================

def evaluate_model(model, n_episodes=EVAL_EPISODES):
    rows = []
    env.training = False
    env.norm_reward = False
    for episode in range(n_episodes):
        obs = env.reset()
        done = np.array([False])
        total_reward = 0.0
        steps = 0
        last_info = {}
        while not bool(done[0]):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, infos = env.step(action)
            total_reward += float(reward[0])
            steps += 1
            last_info = infos[0]
        terminal = {}
        try:
            terminal = get_episode_terminal_metrics(last_info)
        except Exception as exc:
            terminal = {"terminal_metrics_error": repr(exc)}
        rows.append({
            "episode": episode,
            "steps": steps,
            "total_reward": total_reward,
            **terminal,
        })
    return pd.DataFrame(rows)

results = evaluate_model(model)
display(results)
print(results.mean(numeric_only=True))

In [ ]:
# ==============================================
# 7) Save artifacts
# ==============================================

model_path = output_root / "ppo_friend_dmlpa_minimal.zip"
vecnorm_path = output_root / "vecnormalize.pkl"
results_path = output_root / "evaluation_results.csv"
manifest_path = output_root / "manifest.json"

model.save(str(model_path))
env.save(str(vecnorm_path))
results.to_csv(results_path, index=False)

manifest = {
    "notebook": "friend_dmlpa_minimal_colab_kaggle.ipynb",
    "run_profile": RUN_PROFILE,
    "seed": SEED,
    "frame_stack": FRAME_STACK,
    "features_dim": FEATURES_DIM,
    "reward_mode": REWARD_MODE,
    "risk_level": RISK_LEVEL,
    "stochastic_pt": STOCHASTIC_PT,
    "step_size_hours": STEP_SIZE_HOURS,
    "max_steps": MAX_STEPS,
    "total_timesteps": TOTAL_TIMESTEPS,
    "eval_episodes": EVAL_EPISODES,
    "n_steps": N_STEPS,
    "batch_size": BATCH_SIZE,
    "n_epochs": N_EPOCHS,
    "device": DEVICE,
    "cuda_available": torch.cuda.is_available(),
    "training_seconds": elapsed_seconds,
    "model_path": str(model_path),
    "vecnormalize_path": str(vecnorm_path),
    "results_path": str(results_path),
}
manifest_path.write_text(json.dumps(manifest, indent=2))

print("saved model:", model_path)
print("saved vecnormalize:", vecnorm_path)
print("saved results:", results_path)
print("saved manifest:", manifest_path)